# Model 3 - XGBoost
**Owner:** _(your name)_  |  **Branch:** `model/xgboost`

Gradient boosting: each tree corrects the previous trees' errors. **Extra experiment:** learning-rate vs number-of-trees trade-off + feature importance.

## 1. Setup and data
Same split, same CV folds and same metrics for every model (from `src/common.py`).

In [ ]:
import sys; sys.path.append("../src")
import warnings; warnings.filterwarnings("ignore")
from common import *
NAME = "XGBoost"

In [ ]:
X_train, X_test, y_train, y_test, g_train, cv, class_names = load_and_split()
print(X_train.shape, X_test.shape, class_names)
pd.Series(y_train).value_counts(normalize=True).sort_index().round(3)   # class balance

## 2. Model pipeline
Preprocessing is **inside** the pipeline, so it is fitted on training folds only (no leakage). Imbalance is handled with class weights rather than by changing the data.

In [ ]:
pipe = Pipeline([("pre", make_preprocessor(X_train, scale=False)),
                 ("m", BalancedXGB(n_estimators=300, learning_rate=0.1, max_depth=6,
                                   eval_metric="mlogloss", n_jobs=-1, random_state=RANDOM_STATE))])

## 3. Baseline (untuned) with stratified cross-validation
Headline metric: **macro-F1**. Compare `train_macro_f1` with `cv_macro_f1`: a big gap means overfitting.

In [ ]:
baseline = baseline_cv(pipe, X_train, y_train, g_train, cv)
pd.Series(baseline).round(3)

## 4. Hyperparameter tuning (RandomizedSearchCV, scored on macro-F1)

In [ ]:
space = {"m__max_depth": [3, 4, 6, 8], "m__learning_rate": [0.03, 0.05, 0.1, 0.2],
         "m__n_estimators": [200, 400, 600], "m__subsample": [0.7, 0.85, 1.0],
         "m__colsample_bytree": [0.6, 0.8, 1.0], "m__min_child_weight": [1, 3, 5]}
search = tune(pipe, space, X_train, y_train, g_train, cv, n_iter=15)
print("best CV macro-F1:", round(search.best_score_, 3))
search.best_params_

## 5. Final evaluation and saving
The test set is only **reported** here. The final model is chosen later by CV score in `compare_models.ipynb`.

In [ ]:
result = finalize(NAME, baseline, search, X_test, y_test, class_names)

## 6. Extra experiment - learning rate vs number of trees
A small learning rate needs more trees but usually generalises better. Compare CV macro-F1 for a few combinations.

In [ ]:
rows = []
for lr_, n in [(0.3, 100), (0.1, 300), (0.05, 600)]:
    p = Pipeline([("pre", make_preprocessor(X_train, scale=False)),
                  ("m", BalancedXGB(n_estimators=n, learning_rate=lr_, max_depth=4,
                                    eval_metric="mlogloss", n_jobs=-1, random_state=RANDOM_STATE))])
    r = baseline_cv(p, X_train, y_train, g_train, cv)
    rows.append({"learning_rate": lr_, "n_estimators": n, "train": r["train_macro_f1"], "cv": r["cv_macro_f1"]})
display(pd.DataFrame(rows).round(3))

best = search.best_estimator_
imp = pd.Series(best.named_steps["m"].feature_importances_, index=transformed_feature_names(best))
imp.sort_values().tail(15).plot(kind="barh", figsize=(7, 5), title="Top 15 features (XGBoost gain)")
plt.tight_layout(); plt.savefig(FIGS_DIR / "xgboost_importance.png", dpi=150); plt.show()

## 7. Be ready to explain (viva)
- Boosting vs bagging: how is XGBoost different from a random forest?
- What do `learning_rate`, `max_depth`, `subsample` and `colsample_bytree` do?
- Why did you use sample weights for imbalance (multiclass has no class_weight)?
- How do you know it is not over-fitting (train vs CV gap)?
- Why might it beat logistic regression on this data?